In [0]:
import os
import sys
from pathlib import Path

import mlflow

username = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
)

project_root = Path(
    f"/Workspace/Users/{username}/finance-timeseries-ml-platform"
)

os.chdir(project_root)
sys.path.insert(0, str(project_root / "src"))

from finance_ml.models.registry import (
    register_logged_model,
    resolve_logged_model_uri,
)

mlflow.set_registry_uri("databricks-uc")

In [0]:
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS finance_ml.models
""")

print("Model registry schema is ready.")

In [0]:
from mlflow.tracking import MlflowClient
import pandas as pd

# Listing all LoggedModels in the selected MLflow experiment.
tracking_client = MlflowClient()

logged_models = []
page_token = None

EXPERIMENT_ID = "2355252254662315"

while True:
    page = tracking_client.search_logged_models(
        experiment_ids=[EXPERIMENT_ID],
        max_results=50,
        page_token=page_token,
    )

    logged_models.extend(page)

    page_token = page.token

    if not page_token:
        break

# Preparing a readable model inventory.
model_inventory = pd.DataFrame(
    [
        {
            "Model ID": model.model_id,
            "Model name": model.name,
            "Source run ID": model.source_run_id,
            "Status": model.status,
            "Model URI": f"models:/{model.model_id}",
        }
        for model in logged_models
    ]
)

print(f"Total LoggedModels: {len(model_inventory)}")

display(model_inventory)

In [0]:
# Automatically selecting the final model

EXPERIMENT_ID = "2355252254662315"

REGISTERED_MODEL_NAME = (
    "finance_ml.models.lstm_risk_classifier"
)

selected_model_uri = resolve_logged_model_uri(
    experiment_id=EXPERIMENT_ID,
    use_final_tag=True,
)

print("Selected MLflow model:", selected_model_uri)

In [0]:
# Registering the model in Unity Catalog

registered_version = register_logged_model(
    model_uri=selected_model_uri,
    registered_model_name=REGISTERED_MODEL_NAME,
)

print("Registered model:", registered_version.name)
print("Version:", registered_version.version)
print("Source:", registered_version.source)

In [0]:
# Verifying inference from the registered model

import mlflow
import numpy as np

registered_model_uri = (
    f"models:/{REGISTERED_MODEL_NAME}/{registered_version.version}"
)

# Loading the model using its Unity Catalog version.
uc_model = mlflow.keras.load_model(registered_model_uri)

# Loading the original MLflow LoggedModel.
source_model = mlflow.keras.load_model(selected_model_uri)

# Generating reproducible sample inputs for verification.
rng = np.random.default_rng(42)

x_check = rng.normal(
    size=(5, 7, 41)    # Test Should pass when the number of features is 41
).astype(np.float32)

uc_predictions = uc_model.predict(x_check, verbose=0)
source_predictions = source_model.predict(x_check, verbose=0)

np.testing.assert_allclose(
    uc_predictions,
    source_predictions,
    rtol=1e-5,
    atol=1e-6,
)

print("Registered model URI:", registered_model_uri)
print("Prediction shape:", uc_predictions.shape)
print("UC model inference verification: PASSED")